# Smart Task Manager — LLM-Powered Description Pipeline

This notebook adds one thing to the existing project: an AI cleanup step for the
task **`description`** field, wired straight into the project's own
`TaskCreate` / `TaskUpdate` Pydantic schemas.

**Scope, on purpose:**
- Only `description` (free text a user typed) goes to the LLM. `title`, `priority`,
  `status`, and `deadline` are never sent to the model — they're structured fields
  the UI/API already validates.
- The LLM is instructed to return **only** the cleaned description text — no
  preamble, no quotes, no "Here's your cleaned description:".
- Whatever comes back is still run through `TaskCreate`/`TaskUpdate` — the LLM is
  not a trusted source of valid data, it's just a text-cleanup step upstream of
  the same validation every other client of this API goes through.

**Flow:** `raw description` → LLM cleanup → defensive text checks → build
`TaskCreate`/`TaskUpdate` → Pydantic validates → (optionally) save via the
existing `crud.py`.

Run cells top to bottom. No code in `app/` is modified — this notebook only
*imports* the existing schemas/models/crud and adds a pipeline on top.


In [ ]:
import sys, os, re
from pathlib import Path

# Make `app.*` importable regardless of whether this notebook is opened from
# the project root or from a subfolder.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "app").exists() and (PROJECT_ROOT.parent / "app").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)  # so the sqlite:///./tasks.db path in database.py resolves correctly

from pydantic import ValidationError
from app.schemas import TaskCreate, TaskUpdate

print(f"Project root: {PROJECT_ROOT}")
print("Imported TaskCreate/TaskUpdate from the existing app.schemas module (unmodified).")


## 1. The LLM cleanup call

`call_llm_for_description` sends **only the raw description string** to the
Anthropic API, with a system prompt that demands raw text back — nothing else.

If `ANTHROPIC_API_KEY` isn't set, this falls back to a small deterministic
**mock cleaner** (whitespace/casing normalization) so the rest of the pipeline
— the part this task is actually about — can still be run and demonstrated
without a key or network access. Swap in a real key any time; nothing else
in the pipeline changes.


In [ ]:
import requests

ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY")
ANTHROPIC_MODEL = os.environ.get("ANTHROPIC_MODEL", "claude-sonnet-5")
ANTHROPIC_API_URL = "https://api.anthropic.com/v1/messages"

CLEAN_DESCRIPTION_SYSTEM_PROMPT = (
    "You clean up a single task description for a to-do app.\n"
    "Rules:\n"
    "- Fix spelling, grammar, and punctuation.\n"
    "- Normalize whitespace (no double spaces, no leading/trailing spaces).\n"
    "- Preserve the original meaning and length - do not invent new tasks, dates, or details.\n"
    "- Do not add a title, bullet points, markdown, or commentary.\n"
    "- Output ONLY the cleaned description text. No preamble, no quotes, no "
    "explanation, no \"Here is...\" - just the raw description text and nothing else."
)


class LLMDescriptionError(RuntimeError):
    """Raised when the LLM step fails, or returns something unusable."""


def _mock_clean_description(raw_description: str) -> str:
    """
    Offline stand-in for the LLM, used only when ANTHROPIC_API_KEY is unset.
    Just enough cleanup (whitespace/casing) to exercise the rest of the
    pipeline end-to-end without a key or network access.
    """
    text = re.sub(r"\s+", " ", raw_description).strip()
    if text:
        text = text[0].upper() + text[1:]
    return text


def call_llm_for_description(raw_description: str) -> str:
    """
    Send ONLY the raw description to the LLM. Returns the raw text response.
    Raises LLMDescriptionError on any failure (network, empty input, empty output).
    """
    if not raw_description or not raw_description.strip():
        raise LLMDescriptionError("Cannot clean an empty description.")

    if not ANTHROPIC_API_KEY:
        return _mock_clean_description(raw_description)

    try:
        response = requests.post(
            ANTHROPIC_API_URL,
            headers={
                "x-api-key": ANTHROPIC_API_KEY,
                "anthropic-version": "2023-06-01",
                "content-type": "application/json",
            },
            json={
                "model": ANTHROPIC_MODEL,
                "max_tokens": 512,
                "system": CLEAN_DESCRIPTION_SYSTEM_PROMPT,
                # Only the description text is sent - nothing else about the task.
                "messages": [{"role": "user", "content": raw_description}],
            },
            timeout=30,
        )
        response.raise_for_status()
    except requests.RequestException as exc:
        raise LLMDescriptionError(f"LLM request failed: {exc}") from exc

    data = response.json()
    blocks = data.get("content", [])
    cleaned = "".join(b.get("text", "") for b in blocks if b.get("type") == "text").strip()

    if not cleaned:
        raise LLMDescriptionError("LLM returned an empty response.")

    return cleaned


print("Mode:", "LIVE (calling Anthropic API)" if ANTHROPIC_API_KEY else "MOCK (no ANTHROPIC_API_KEY set)")


## 2. Defensive checks on the LLM's output

The system prompt says "raw text only, no commentary" — but prompts aren't
guarantees. Before we ever hand the text to Pydantic, we do two cheap,
deterministic checks:

- **Strip wrapping quotes** the model sometimes adds (`"buy milk"` → `buy milk`).
- **Detect leaked commentary** (`"Sure! Here's the cleaned description: ..."`)
  and fail loudly instead of silently saving the commentary as the description.

This is a belt-and-braces layer, not a replacement for schema validation —
it just catches a specific, predictable failure mode before it reaches Pydantic.


In [ ]:
_COMMENTARY_PATTERNS = [
    r"^here('s| is)\\b",
    r"^sure[,!]",
    r"^certainly[,!]",
    r"^i (have|'ve) (cleaned|rewritten|revised)",
    r"^cleaned description:?",
]


def _strip_wrapping_quotes(text: str) -> str:
    if len(text) >= 2 and text[0] == text[-1] and text[0] in "\"'":
        return text[1:-1].strip()
    return text


def looks_like_commentary(text: str) -> bool:
    lowered = text.strip().lower()
    return any(re.match(pattern, lowered) for pattern in _COMMENTARY_PATTERNS)


def clean_description(raw_description: str) -> str:
    """
    Full description-cleaning step used by the pipeline below:
    call the LLM, then defensively strip quotes / reject leaked commentary.
    """
    cleaned = call_llm_for_description(raw_description)
    cleaned = _strip_wrapping_quotes(cleaned).strip()

    if looks_like_commentary(cleaned):
        raise LLMDescriptionError(
            f"LLM returned commentary instead of raw description text: {cleaned!r}"
        )

    return cleaned


# quick self-check of the guard logic itself
assert looks_like_commentary("Sure! Here's the cleaned description: buy milk")
assert not looks_like_commentary("Buy milk and eggs on the way home")
print("Guard logic OK.")


## 3. The pipeline: clean → validate against `TaskCreate` / `TaskUpdate`

Two thin wrapper functions. Neither one touches the database or the API layer —
they just produce a **validated Pydantic object**, exactly like `POST /tasks`
or `PUT /tasks/{id}` would end up with after FastAPI's own request validation.
Only `raw_description` is AI-processed; every other field passes through as-is.


In [ ]:
from typing import Optional


def build_task_create_with_ai_description(
    *,
    title: str,
    raw_description: str,
    priority: str,
    status: str = "Pending",
    deadline: Optional[str] = None,
) -> TaskCreate:
    """
    Build a validated TaskCreate. Only `raw_description` is sent to the LLM;
    title/priority/status/deadline are passed straight through and validated
    by TaskCreate itself, same as any other client of this schema.
    """
    cleaned_description = clean_description(raw_description)
    return TaskCreate(
        title=title,
        description=cleaned_description,
        priority=priority,
        status=status,
        deadline=deadline,
    )


def build_task_update_with_ai_description(*, raw_description: str, **other_fields) -> TaskUpdate:
    """Same idea for partial updates (PUT /tasks/{id})."""
    cleaned_description = clean_description(raw_description)
    return TaskUpdate(description=cleaned_description, **other_fields)


## 4. Demo — happy path (`TaskCreate`)

A deliberately messy, typo-ridden description goes in; a validated
`TaskCreate` comes out.


In [ ]:
raw = (
    "  need   to    finish  the quarterly  report asap!! forgot to add "
    "the appendix also fix typos in intro section  "
)

task = build_task_create_with_ai_description(
    title="Finish quarterly report",
    raw_description=raw,
    priority="High",
)

print("RAW DESCRIPTION:  ", repr(raw))
print("CLEANED DESCRIPTION:", repr(task.description))
print()
print("Validated TaskCreate:", task.model_dump())


## 5. Demo — partial update (`TaskUpdate`)

Same idea, but only `description` and `status` are being changed — the rest
stays untouched, matching how `crud.update_task` applies partial updates.


In [ ]:
update = build_task_update_with_ai_description(
    raw_description="switch   status to in progress   pls, waiting on legal review rn",
    status="In Progress",
)

print("Validated TaskUpdate (fields actually set):", update.model_dump(exclude_unset=True))


## 6. Demo — failure handling

Two failure modes, both handled explicitly rather than silently:

1. **Schema validation fails** — e.g. `title` too short. The cleaned
   description was fine; a *different* field is what's invalid. This proves
   the pipeline doesn't bypass `TaskCreate`'s existing rules.
2. **LLM step fails outright** — empty input, or (simulated here) a model
   that ignores instructions and returns commentary instead of raw text.


In [ ]:
# 6a. Pydantic validation catches an invalid field unrelated to the AI step
try:
    build_task_create_with_ai_description(title="x", raw_description="fix bug", priority="High")
except ValidationError as e:
    print("Caught ValidationError (title too short), as expected:")
    print(e)


In [ ]:
# 6b. Empty description never reaches the LLM
try:
    clean_description("   ")
except LLMDescriptionError as e:
    print("Caught LLMDescriptionError (empty description):", e)


In [ ]:
# 6c. Simulate a misbehaving LLM that leaks commentary despite instructions.
# We temporarily swap out call_llm_for_description to return a bad response,
# and confirm the commentary guard (section 2) catches it before it can
# reach TaskCreate/TaskUpdate at all.
_real_call_llm_for_description = call_llm_for_description


def _misbehaving_llm(raw_description: str) -> str:
    return "Sure! Here's the cleaned description: buy groceries"


call_llm_for_description = _misbehaving_llm
try:
    clean_description("buy groceries pls")
except LLMDescriptionError as e:
    print("Caught LLMDescriptionError (LLM added commentary):", e)
finally:
    call_llm_for_description = _real_call_llm_for_description  # restore the real one


## 7. Demo — end to end, saved through the existing `crud`/DB layer

To confirm this is actually aligned with the backend (not just a standalone
Pydantic exercise), the validated `TaskCreate` from section 4 gets saved
through the project's own `app.crud.create_task`, into the same `tasks.db`
the FastAPI app uses.


In [ ]:
from app.database import SessionLocal, engine, Base
from app import crud

Base.metadata.create_all(bind=engine)  # no-op if tasks.db / table already exist

db = SessionLocal()
try:
    saved = crud.create_task(db, task)  # `task` is the TaskCreate built in section 4
    print(f"Saved task id={saved.id}")
    print(f"  title:       {saved.title}")
    print(f"  description: {saved.description}")
    print(f"  priority:    {saved.priority} | status: {saved.status}")
finally:
    db.close()


## What this demonstrates

- The AI step touches **only** `description` — never `title`, `priority`,
  `status`, or `deadline`.
- The system prompt asks for raw text only; a defensive check (section 2)
  catches the model if it doesn't comply, before Pydantic ever sees it.
- The cleaned text is validated the exact same way any other input to this
  API is validated — through the project's real `TaskCreate`/`TaskUpdate`,
  unmodified.
- It round-trips through the real `crud.py` / `tasks.db`, not a mock.

## Extending this next

1. **Wire it into the API itself.** Right now this pipeline lives in the
   notebook. The natural next step is a helper in `app/utils.py` (e.g.
   `ai_clean_description(raw: str) -> str`) that `routers/tasks.py` calls
   before handing off to `crud.create_task` / `crud.update_task` — likely
   behind an opt-in flag or query param (`POST /tasks?ai_clean=true`) so
   existing clients aren't forced through an extra network call.
2. **Cache/skip redundant calls.** If a description hasn't changed since the
   last clean, skip the LLM call — cheaper and faster, and avoids rewriting
   text a user may have deliberately re-approved as-is.
3. **Make failure behavior a product decision.** Right now `LLMDescriptionError`
   just aborts. Decide (and make explicit) what should happen instead: fall back
   to the raw description unmodified, retry once, or reject the request — this
   changes user experience a lot and shouldn't be an accident of implementation.
